In [1]:
import os
import pandas as pd
import numpy as np

# 1. Tentukan nama folder tempat file CSV Anda berada
nama_folder = "../Dataset"

# 2. Baca semua file CSV di dalam folder secara otomatis ke dalam dictionary 'tables'
tables = {}
for file in os.listdir(nama_folder):
    if file.endswith('.csv'):
        # Mengambil nama file tanpa ekstensi '.csv' sebagai key (misal: 'kunjungan')
        nama_tabel = os.path.splitext(file)[0] 
        path_file = os.path.join(nama_folder, file)
        
        # Load ke pandas
        tables[nama_tabel] = pd.read_csv(path_file)
        print(f"✔ {nama_tabel:<11} {len(tables[nama_tabel]):>7,} baris | {tables[nama_tabel].shape[1]} kolom")

# Alias untuk tabel yang dipakai di analisis
df_customers = tables["customers"]
df_sellers   = tables["sellers"]
df_products  = tables["products"]
df_orders    = tables["orders"]
df_order_items = tables["order_items"]
df_product_category = tables["product_category_name_translation"]

✔ customers    99,441 baris | 5 kolom
✔ geolocation 1,000,163 baris | 5 kolom
✔ orders       99,441 baris | 8 kolom
✔ order_items 112,650 baris | 7 kolom
✔ order_payments 103,886 baris | 5 kolom
✔ order_reviews  99,224 baris | 7 kolom
✔ products     32,951 baris | 9 kolom
✔ product_category_name_translation      71 baris | 2 kolom
✔ sellers       3,095 baris | 4 kolom


In [2]:
# Membuat dataframe audit
audit = pd.DataFrame({
    "baris": {n: len(t) for n, t in tables.items() if n in ["customers", "order_items", "sellers", "orders","product_category","products"]},
}).assign(
    kolom=lambda d: [tables[n].shape[1] for n in d.index],
    total_null=lambda d: [int(tables[n].isna().sum().sum()) for n in d.index],
)
display(audit)

print("\nNilai kosong per kolom (hanya yang > 0):")
for n in ["Customers", "order_items", "sellers", "orders","product_category","products"]:
    # Proteksi jika file ternyata tidak ada di folder agar kode tidak error
    if n in tables: 
        nulls = tables[n].isna().sum()
        nulls = nulls[nulls > 0]
        if len(nulls):
            print(f"\n[{n}]")
            print(nulls.to_string())


,baris,kolom,total_null
customers,99441,5,0
order_items,112650,7,0
orders,99441,8,4908
products,32951,9,2449
sellers,3095,4,0



Nilai kosong per kolom (hanya yang > 0):

[orders]
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

[products]
product_id                      1
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2


In [3]:
base_1 = df_orders.merge(df_customers, how='left', on='customer_id')
base_1.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [4]:
base_2 = df_order_items.merge(df_sellers, how='left', on='seller_id')
base_2.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR


In [5]:
main_base_geo_and_loc = base_2.merge(base_1, how="left", on="order_id").assign(is_same_state=lambda d: d['customer_state'] == d['seller_state'])
main_base_geo_and_loc.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,...,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,is_same_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,...,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,...,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,True
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,...,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,True
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,...,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,...,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,False


In [6]:
main_base_spec_products = main_base_geo_and_loc.merge(df_products, how="left", on="product_id").assign(product_volume_cm3=lambda d: d['product_length_cm'] * d['product_height_cm'] * d['product_width_cm']).merge(df_product_category[['product_category_name', 'product_category_name_english']], how="left", on="product_category_name")


In [7]:
main_base_spec_products.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,...,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,...,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,3528.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,...,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,60000.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,...,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,14157.0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,...,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,2400.0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,...,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,42000.0,garden_tools


In [8]:
main_base_spec_products['order_purchase_timestamp'] = pd.to_datetime(main_base_spec_products['order_purchase_timestamp'])
main_base_spec_products['purchase_dayofweek'] = main_base_spec_products['order_purchase_timestamp'].dt.dayofweek
main_base_spec_products['purchase_hour'] = main_base_spec_products['order_purchase_timestamp'].dt.hour

In [9]:
main_base_spec_products = main_base_spec_products.assign(is_late=lambda d: d['order_delivered_customer_date'] > d['order_estimated_delivery_date'])

In [10]:
main_base_spec_products.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,...,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3,product_category_name_english,purchase_dayofweek,purchase_hour,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,...,4.0,650.0,28.0,9.0,14.0,3528.0,cool_stuff,2,8,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,...,2.0,30000.0,50.0,30.0,40.0,60000.0,pet_shop,2,10,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,...,2.0,3050.0,33.0,13.0,33.0,14157.0,furniture_decor,6,14,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,...,1.0,200.0,16.0,10.0,15.0,2400.0,perfumery,2,10,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,...,1.0,3750.0,35.0,40.0,30.0,42000.0,garden_tools,5,13,False


In [29]:
main_df = main_base_spec_products[['customer_state', 'seller_state', 'is_same_state','freight_value','product_weight_g', 'product_volume_cm3','price', 'product_category_name_english', 'purchase_dayofweek', 'purchase_hour', 'is_late']]
main_df[['is_same_state', 'is_late']] = main_df[['is_same_state', 'is_late']].astype(int)
main_df.head()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8840\3356570036.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_df[['is_same_state', 'is_late']] = main_df[['is_same_state', 'is_late']].astype(int)


,customer_state,seller_state,is_same_state,freight_value,product_weight_g,product_volume_cm3,price,product_category_name_english,purchase_dayofweek,purchase_hour,is_late
0,RJ,SP,0,13.29,650.0,3528.0,58.90,cool_stuff,2,8,0
1,SP,SP,1,19.93,30000.0,60000.0,239.90,pet_shop,2,10,0
2,MG,MG,1,17.87,3050.0,14157.0,199.00,furniture_decor,6,14,0
3,SP,SP,1,12.79,200.0,2400.0,12.99,perfumery,2,10,0
4,SP,PR,0,18.14,3750.0,42000.0,199.90,garden_tools,5,13,0


Preprocessing

In [30]:
main_df.isnull().sum()

customer_state                      0
seller_state                        0
is_same_state                       0
freight_value                       0
product_weight_g                   19
product_volume_cm3                 19
price                               0
product_category_name_english    1629
purchase_dayofweek                  0
purchase_hour                       0
is_late                             0
dtype: int64

In [31]:
main_df.dropna(subset=['product_weight_g', 'product_volume_cm3'], inplace=True)
main_df['product_category_name_english'] = main_df['product_category_name_english'].fillna('unknown')
main_df.isnull().sum()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8840\831937140.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_df.dropna(subset=['product_weight_g', 'product_volume_cm3'], inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_8840\831937140.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_df['product_category_name_english'] = main_df['product_category_name_english'].fillna('unknown')


customer_state                   0
seller_state                     0
is_same_state                    0
freight_value                    0
product_weight_g                 0
product_volume_cm3               0
price                            0
product_category_name_english    0
purchase_dayofweek               0
purchase_hour                    0
is_late                          0
dtype: int64

In [32]:
main_df.duplicated().sum()

11104

In [33]:
main_df.drop_duplicates(inplace=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8840\3616089854.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_df.drop_duplicates(inplace=True)


In [34]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 101527 entries, 0 to 112649
Data columns (total 11 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   customer_state                 101527 non-null  object 
 1   seller_state                   101527 non-null  object 
 2   is_same_state                  101527 non-null  int32  
 3   freight_value                  101527 non-null  float64
 4   product_weight_g               101527 non-null  float64
 5   product_volume_cm3             101527 non-null  float64
 6   price                          101527 non-null  float64
 7   product_category_name_english  101527 non-null  object 
 8   purchase_dayofweek             101527 non-null  int64  
 9   purchase_hour                  101527 non-null  int64  
 10  is_late                        101527 non-null  int32  
dtypes: float64(4), int32(2), int64(2), object(3)
memory usage: 8.5+ MB


In [35]:
main_df.describe()

,is_same_state,freight_value,product_weight_g,product_volume_cm3,price,purchase_dayofweek,purchase_hour,is_late
count,101527.000000,101527.000000,101527.000000,101527.000000,101527.000000,101527.000000,101527.000000,101527.000000
mean,0.358407,20.161431,2097.005713,15198.226216,124.808535,2.759365,14.776828,0.078491
std,0.479535,15.927637,3759.657378,23436.737756,189.451746,1.968109,5.333027,0.268945
min,0.000000,0.000000,0.000000,168.000000,0.850000,0.000000,0.000000,0.000000
25%,0.000000,13.210000,300.000000,2816.000000,40.500000,1.000000,11.000000,0.000000
50%,0.000000,16.360000,700.000000,6384.000000,79.000000,3.000000,15.000000,0.000000
75%,1.000000,21.260000,1800.000000,18240.000000,139.000000,4.000000,19.000000,0.000000
max,1.000000,409.680000,40425.000000,296208.000000,6735.000000,6.000000,23.000000,1.000000


In [42]:
main_df_encoded = pd.get_dummies(main_df, columns=['customer_state', 'seller_state', 'product_category_name_english'], drop_first=True)

In [44]:
x = main_df_encoded.drop(columns=['is_late'])
y = main_df_encoded['is_late']

In [45]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)